In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr

import zipfile as zp          # used for unzipping ppi files
from pathlib import Path      # used to play with pathnames to save 
from datetime import datetime # used to manipulate time :)

import wradlib as wr          # used for having fun with radar data

from PIL import Image         # used for creating gif loops
import os                     # used for retrieving file names

import h5py                   # used for reading .h5 files (Radar Level 1 data)
import h5netcdf               # used for converting .h5 files to NetCDF

# TAKEN FROM "Part4IntroductionToGridding"
import cartopy.crs as ccrs
import pyart
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.ticker as mticker

In [ ]:
# FUNCTION
#pcolormesh but takes 1D X and Y coordinates for centres of the pixels

def pcolormeshC(x_centers, y_centers, z, ax=None,
                            shading='auto', **pcolor_kwargs):
    """
    Create a pcolormesh from a 2D array and 1D coordinate-center arrays.

    Parameters
    ----------
    x_centers : 1D array
        X coordinates of cell centers (length = number of columns in z)
    y_centers : 1D array
        Y coordinates of cell centers (length = number of rows in z)
    z : 2D array
        Data array with shape (len(y_centers), len(x_centers))
    ax : matplotlib.axes.Axes, optional
        Existing axis to draw on
    shading : str
        Passed to pcolormesh (default: 'auto')
    **pcolor_kwargs
        Extra kwargs passed to pcolormesh

    Returns
    -------
    pcm : QuadMesh
        The pcolormesh object
    """

    x_centers = np.asarray(x_centers)
    y_centers = np.asarray(y_centers)
    z = np.asarray(z)

    if z.shape != (len(y_centers), len(x_centers)):
        raise ValueError(
            f"z shape {z.shape} does not match "
            f"(len(y_centers), len(x_centers)) = "
            f"({len(y_centers)}, {len(x_centers)})"
        )

    # Convert centers -> edges
    def centers_to_edges(c):
        dc = np.diff(c)

        edges = np.empty(len(c) + 1)

        # Interior edges
        edges[1:-1] = c[:-1] + dc / 2

        # Extrapolate outer edges
        edges[0] = c[0] - dc[0] / 2
        edges[-1] = c[-1] + dc[-1] / 2

        return edges

    x_edges = centers_to_edges(x_centers)
    y_edges = centers_to_edges(y_centers)

    if ax is None:
        fig, ax = plt.subplots()

    pcm = ax.pcolormesh(
        x_edges,
        y_edges,
        z,
        shading=shading,
        **pcolor_kwargs
    )

    ax.set_xlabel("X")
    ax.set_ylabel("Y")

    return pcm

In [ ]:
# THIS BLOCK IS WHERE THE USER PUTS INFO ABOUT THE RADAR

# the reference number for the radar location (ie 20 is Mackay)
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)
GridOrPPI = 'ppi'

# the day in consideration (YYYYMMDD) and time (hhmmss)
# ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 2
RadarDay   = 14

RadarFileDate  = str(RadarYear).zfill(4) + str(RadarMonth).zfill(2) + str(RadarDay).zfill(2)
RadarFileTime =   '000000'

In [ ]:
# THIS BLOCK TAKES A ZIPPED FOLDER FROM /G/DATA/ AND UNZIPS IT TO A FOLDER "nzippedRadarFiles" IN SCRATCH

# PATH NAMES
# ZippedFolder = '/g/data/rq0/level_1b/22/ppi/2024/'
ZippedFolder = '/g/data/rq0/level_1b/' + RadarIDno + '/' + GridOrPPI + '/' + str(RadarYear) + '/'
ZippedFile   =  RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + '.zip'

# place where the zipped file lives
ZippedPath = ZippedFolder + ZippedFile
# place to extract the files to
ExtractToDirectory = Path('/scratch/v46/sg3241/tmp/UnzippedRadarFiles/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + '/')

# Create the extraction folder (only if you haven't already)
if not Path(ExtractToDirectory).exists():
    ExtractToDirectory.mkdir(parents=True, exist_ok=True)

    # actually do the unzipping
    with zp.ZipFile(ZippedPath, 'r') as ZipReference:
        ZipReference.extractall(ExtractToDirectory)

    print('Done')
else:
    print('Unzipped Folder Already Exists')

In [ ]:
# READ IN THE DATA TO A PYART FILE

# the path of where the UNZIPPED radar data now live
RadarFolder = '/scratch/v46/sg3241/tmp/UnzippedRadarFiles/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + '/'
RadarFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + GridOrPPI + '.nc'


# open the radar data 
RadarPyArt = pyart.io.read(RadarFolder + RadarFile, delay_field_loading = True)


# FOR CHECKING
# open the radar data into an xarray data structure
RadarDataStruct = xr.open_dataset(RadarFolder + RadarFile, engine='netcdf4')


In [ ]:
# convert the ppi data to a grid
grids = pyart.map.grid_from_radars(RadarPyArt,(31,301,301),
                   ((0.,15000.),(-150000.,150000.),(-150000.,150000.)),
                   refl_field='corrected_reflectivity', weighting_function='Barnes2')

# convert to an xarray data frame
xgrids = grids.to_xarray()

In [ ]:
# ALL IN ONE LOOP!


# THIS BLOCK IS WHERE THE USER PUTS INFO ABOUT THE RADAR

# the reference number for the radar location (ie 20 is Mackay)
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)
GridOrPPI = 'ppi'

# the day in consideration (YYYYMMDD) and time (hhmmss)
# ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 2
RadarDay   = 14

RadarFileDate  = str(RadarYear).zfill(4) + str(RadarMonth).zfill(2) + str(RadarDay).zfill(2)
RadarFileDatePrint = RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] # add a string of format YYYY-MM-DD for printing



# THIS BLOCK TAKES A ZIPPED FOLDER FROM /G/DATA/ AND UNZIPS IT TO A FOLDER "nzippedRadarFiles" IN SCRATCH

# PATH NAMES
# ZippedFolder = '/g/data/rq0/level_1b/22/ppi/2024/'
ZippedFolder = '/g/data/rq0/level_1b/' + RadarIDno + '/' + GridOrPPI + '/' + str(RadarYear) + '/'
ZippedFile   =  RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + '.zip'

# place where the zipped file lives
ZippedPath = ZippedFolder + ZippedFile
# place to extract the files to
ExtractToDirectory = Path('/scratch/v46/sg3241/tmp/UnzippedRadarFiles/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + '/')

# Create the extraction folder (only if you haven't already)
if not Path(ExtractToDirectory).exists():
    ExtractToDirectory.mkdir(parents=True, exist_ok=True)

    # actually do the unzipping
    with zp.ZipFile(ZippedPath, 'r') as ZipReference:
        ZipReference.extractall(ExtractToDirectory)

    print('Done')
else:
    print('Unzipped Folder Already Exists')


# loop over every 5 min period in the day
for houri in range(18,24):
    for mini in range(0,60,5):
        RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
        RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] # add a string of format hh:mm:ss for printing
        print('working on ' + RadarFileTimePrint)

        # READ IN THE DATA TO A PYART FILE
        
        # the path of where the UNZIPPED radar data now live
        RadarFolder = '/scratch/v46/sg3241/tmp/UnzippedRadarFiles/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + '/'
        RadarFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + GridOrPPI + '.nc'
        
        
        # open the radar data 
        RadarPyArt = pyart.io.read(RadarFolder + RadarFile, delay_field_loading = True)
        
        
        # convert the ppi data to a grid
        grids = pyart.map.grid_from_radars(RadarPyArt,(31,301,301),
                           ((0.,15000.),(-150000.,150000.),(-150000.,150000.)),
                           refl_field='corrected_reflectivity', weighting_function='Barnes2')
        
        # convert to an xarray data frame
        xgrids = grids.to_xarray()
        
        
        # Create a plot of the reflectivity for a 5-min period
        
        fig, ax = plt.subplots(figsize=(8,6))
        GridViewer = pcolormeshC(xgrids.x*0.001, xgrids.z, xgrids.corrected_reflectivity[0,:,200,:], ax=ax, cmap='nipy_spectral', vmin=-10, vmax=50)
        # mutiply by 0.001 to get distances in km                                        # THIS SLICE COMES FROM 50 KM NORTH OF THE RADAR TO BETTER SEE THE VOLUME
        
        ax.set_xlabel('East-West Distance [km]')
        ax.set_ylabel('Altitude Above Radar [m]')
        plt.title('Radar Site ' + str(RadarIDno) + ' Reflectivity on ' + RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
                  str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6])
        plt.colorbar(GridViewer, ax=ax, label = 'Reflectivity [dBZ]')
        plt.grid()
        
        PlotType = 'Vert'
        PlotVar  = 'corrected_reflectivity'
        Slice    = '0kmEW'
        
        SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/' + PlotType + '/' + RadarIDno + '/' + RadarFileDate + '/'
        SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + PlotVar + '_' + PlotType + Slice + '.png'
        
        SavePath = SaveFolder + SaveFile
        
        if not Path(SaveFolder).exists():
            print('Creating Folder: ' + SaveFolder)
            Path(SaveFolder).mkdir(parents=True, exist_ok=True)
        
        plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)

In [ ]:
# THIS BLOCK CREATES A GIF FROM THE SAVED RADAR REFLECTIVITY PNG IMAGES
                
# LOADING IMAGES
files = sorted(os.listdir(SaveFolder)) # takes all of the files in the folder in the order they are named
images = [
    Image.open(os.path.join(SaveFolder, f))
    for f in files
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
]

GIFsaveFolder = '/scratch/v46/sg3241/tmp/gifImages/' + PlotType + '/' + RadarIDno + '/' + RadarFileDate + '/'
GIFsaveFile   = RadarIDno + '_' + RadarFileDate + '_' + PlotVar + '_' + PlotType + Slice + '.gif'

GIFsavePath = GIFsaveFolder + GIFsaveFile

if not Path(GIFsaveFolder).exists():
    Path(GIFsaveFolder).mkdir(parents=True, exist_ok=True)

# Save as looping GIF
images[0].save(
    GIFsavePath,
    save_all=True,
    append_images=images[1:],
    duration=200,    # ms per frame
    loop=0,          # 0 = loop forever
)
print('Saved GIF for ' + RadarFileDatePrint)



In [ ]:
# THIS BLOCK IS WHERE THE USER PUTS INFO ABOUT THE RADAR

# the reference number for the radar location (ie 20 is Mackay)
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)
GridOrPPI = 'ppi'

# the day in consideration (YYYYMMDD) and time (hhmmss)
# ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 2
RadarDay   = 14

RadarFileDate  = str(RadarYear).zfill(4) + str(RadarMonth).zfill(2) + str(RadarDay).zfill(2)
RadarFileDatePrint = RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] # add a string of format YYYY-MM-DD for printing


# THIS BLOCK CREATES A GIF FROM THE SAVED RADAR REFLECTIVITY PNG IMAGES

PlotType='CFAD'
PlotVar  = 'corrected_reflectivity'

# # LOADING IMAGES
# files = sorted(os.listdir(SaveFolder)) # takes all of the files in the folder in the order they are named
# images = [
#     Image.open(os.path.join(SaveFolder, f))
#     for f in files
#     if f.endswith(("kmEW.png"))
#     and f.startswith("22_20240214_010000_corrected_reflectivity_Vert")  # only vertical cross section pngs
# ]

SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/CFAD/22/20240214/'

# LOADING IMAGES
files = sorted(os.listdir(SaveFolder))

images = [
    Image.open(os.path.join(SaveFolder, f))
    for f in files
    if f.startswith("22_20240214_")
    and f.endswith("_corrected_reflectivity_CFAD.png")
]

# scratch/v46/sg3241/tmp/pngImages/CFAD/22/20240214/22_20240214_003500_corrected_reflectivity_CFAD.png

GIFsaveFolder = '/scratch/v46/sg3241/tmp/gifImages/' + PlotType + '/' + RadarIDno + '/' + RadarFileDate + '/'
GIFsaveFile   = RadarIDno + '_' + RadarFileDate + '_' + PlotVar + '_' + PlotType + 'SliceAnimation.gif'

GIFsavePath = GIFsaveFolder + GIFsaveFile

if not Path(GIFsaveFolder).exists():
    Path(GIFsaveFolder).mkdir(parents=True, exist_ok=True)

# Save as looping GIF
images[0].save(
    GIFsavePath,
    save_all=True,
    append_images=images[1:],
    duration=200,    # ms per frame
    loop=0,          # 0 = loop forever
)
print('Saved GIF')

In [ ]:
# GIF MAKER
# FOR VERTICAL CROSS SECTION

# THIS BLOCK IS WHERE THE USER PUTS INFO ABOUT THE RADAR

# the reference number for the radar location (ie 20 is Mackay)
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)
GridOrPPI = 'ppi'

# the day in consideration (YYYYMMDD) and time (hhmmss)
# ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 2
RadarDay   = 14

RadarFileDate  = str(RadarYear).zfill(4) + str(RadarMonth).zfill(2) + str(RadarDay).zfill(2)
RadarFileDatePrint = RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] # add a string of format YYYY-MM-DD for printing


# THIS BLOCK CREATES A GIF FROM THE SAVED RADAR REFLECTIVITY PNG IMAGES
                
# LOADING IMAGES
files = sorted(os.listdir(SaveFolder)) # takes all of the files in the folder in the order they are named
images = [
    Image.open(os.path.join(SaveFolder, f))
    for f in files
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
]

SaveFolder = '/cratch/v46/sg3241/tmp/pngImages/Vert/22/20240214/'

GIFsaveFolder = '/scratch/v46/sg3241/tmp/gifImages/' + PlotType + '/' + RadarIDno + '/' + RadarFileDate + '/'
GIFsaveFile   = RadarIDno + '_' + RadarFileDate + '_' + PlotVar + '_' + PlotType + Slice + '.gif'

GIFsavePath = GIFsaveFolder + GIFsaveFile

if not Path(GIFsaveFolder).exists():
    Path(GIFsaveFolder).mkdir(parents=True, exist_ok=True)

# Save as looping GIF
images[0].save(
    GIFsavePath,
    save_all=True,
    append_images=images[1:],
    duration=200,    # ms per frame
    loop=0,          # 0 = loop forever
)
print('Saved GIF for ' + RadarFileDatePrint)